<a href="https://colab.research.google.com/github/bipana-aa/Numerical-Methods/blob/main/Week_2_Lab_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Numerical Methods: Chapter 2 Exercises
**Name:** Bipana Khadka  
**Course:** BSIT400

## CA2.2 Inverse of lower and upper triangular matrices (Gauss-Jordan)

**Idea:** Row-reduce the augmented matrix $[T \mid I]$ to $[I \mid T^{-1}]$. The triangular shape removes half of the elimination work.

**Pseudocode (lower triangular L):**
1. Start with `inv` = identity matrix.
2. For k = 1 to n:
   1. If L[k][k] = 0, stop: the matrix is singular.
   2. Divide row k of `inv` by L[k][k].
   3. For each row i > k: row i of `inv` -= L[i][k] * row k of `inv`.
3. Return `inv`.

**Pseudocode (upper triangular U):** same steps, but sweep k = n down to 1 and eliminate rows i < k.

Cost is about n^3/6 multiplications.

In [ ]:
def inverse_lower(L, tol=1e-12):
    n = len(L)
    inv = [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]
    for k in range(n):
        piv = L[k][k]
        if abs(piv) < tol:
            raise ValueError(f"Zero diagonal entry at ({k+1},{k+1}): matrix is singular")
        for j in range(k + 1):              # divide row k by the pivot
            inv[k][j] /= piv
        for i in range(k + 1, n):           # clear column k below the pivot
            f = L[i][k]
            if f != 0.0:
                for j in range(k + 1):
                    inv[i][j] -= f * inv[k][j]
    return inv


def inverse_upper(U, tol=1e-12):
    n = len(U)
    inv = [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]
    for k in range(n - 1, -1, -1):
        piv = U[k][k]
        if abs(piv) < tol:
            raise ValueError(f"Zero diagonal entry at ({k+1},{k+1}): matrix is singular")
        for j in range(k, n):               # divide row k by the pivot
            inv[k][j] /= piv
        for i in range(k - 1, -1, -1):      # clear column k above the pivot
            f = U[i][k]
            if f != 0.0:
                for j in range(k, n):
                    inv[i][j] -= f * inv[k][j]
    return inv

### Test helpers and a hand-checkable example

In [ ]:
import random

def matmul(A, B):
    n = len(A)
    return [[sum(A[i][k] * B[k][j] for k in range(n)) for j in range(n)] for i in range(n)]

def max_err_from_identity(P):
    n = len(P)
    return max(abs(P[i][j] - (1.0 if i == j else 0.0)) for i in range(n) for j in range(n))

def show(name, M):
    print(name)
    for row in M:
        print("  " + "  ".join(f"{x:9.5f}" for x in row))

L = [[2.0, 0, 0], [1, 4, 0], [3, 5, 8]]
U = [[2.0, 1, 3], [0, 4, 5], [0, 0, 8]]

Li, Ui = inverse_lower(L), inverse_upper(U)
show("L =", L); show("inv(L) =", Li)
print("max |L*inv(L) - I| =", max_err_from_identity(matmul(L, Li)), "\n")
show("U =", U); show("inv(U) =", Ui)
print("max |U*inv(U) - I| =", max_err_from_identity(matmul(U, Ui)))

L =
    2.00000    0.00000    0.00000
    1.00000    4.00000    0.00000
    3.00000    5.00000    8.00000
inv(L) =
    0.50000    0.00000    0.00000
   -0.12500    0.25000    0.00000
   -0.10938   -0.15625    0.12500
max |L*inv(L) - I| = 0.0 

U =
    2.00000    1.00000    3.00000
    0.00000    4.00000    5.00000
    0.00000    0.00000    8.00000
inv(U) =
    0.50000   -0.12500   -0.10938
    0.00000    0.25000   -0.15625
    0.00000    0.00000    0.12500
max |U*inv(U) - I| = 0.0


### Random tests and a singular case

In [ ]:
def random_triangular(n, lower=True):
    M = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if (lower and j <= i) or (not lower and j >= i):
                M[i][j] = random.uniform(-3, 3)
        M[i][i] = random.choice([-1, 1]) * random.uniform(1, 4)
    return M

random.seed(1)
for n in (1, 2, 5, 10, 25):
    L, U = random_triangular(n, True), random_triangular(n, False)
    eL = max_err_from_identity(matmul(L, inverse_lower(L)))
    eU = max_err_from_identity(matmul(U, inverse_upper(U)))
    print(f"n={n:2d}  lower err={eL:.2e}  upper err={eU:.2e}")

try:
    inverse_lower([[1.0, 0.0], [2.0, 0.0]])
except ValueError as e:
    print("\nSingular test:", e)

n= 1  lower err=0.00e+00  upper err=0.00e+00
n= 2  lower err=0.00e+00  upper err=1.11e-16
n= 5  lower err=1.11e-16  upper err=1.67e-16
n=10  lower err=9.99e-16  upper err=3.64e-16
n=25  lower err=1.93e-13  upper err=4.86e-15

Singular test: Zero diagonal entry at (2,2): matrix is singular
